# 6 · Taller 5 — Prototipo de Agente IA para la Universidad (Gemini)
### Plantilla de trabajo

**Complejidad: 🟢 Básica para empezar (puedes subir de complejidad según tu reto)**
**Dependencias base: solo `google-genai`**

Esta plantilla usa el patrón más simple y confiable (el loop manual del notebook 2) para que **ningún equipo se bloquee por instalación** de un framework. Si tu reto necesita RAG o MCP, agrega esas dependencias solo si las necesitas — mira los notebooks 4 y 5 para copiar el bloque de instalación correspondiente.

| Si tu reto necesita... | Copia las dependencias de... |
|---|---|
| Solo herramientas/acciones simples | Este notebook (`google-genai` únicamente) |
| Consultar documentos propios (RAG) | `llamaindex_rag_gemini.ipynb` |
| Exponer herramientas de forma estandarizada | `mcp_servidor_cliente_gemini.ipynb` |
| Memoria conversacional, multi-agente | `langchain_agente_gemini.ipynb` |


## 🎯 Objetivo de aprendizaje

Al terminar este notebook (y el taller) tu equipo va a poder:
- Definir con claridad el problema, el usuario y el criterio de éxito de un agente de IA para un caso real de la universidad.
- Diseñar e implementar las herramientas necesarias para ese agente, siguiendo el mismo patrón visto en los notebooks anteriores.
- Ejecutar casos de prueba y reflexionar críticamente sobre los riesgos y el nivel de autonomía apropiado para su prototipo.


## 📚 Teoría: de la charla al prototipo

Este taller integra todo lo visto en la sesión: el ciclo de un agente (notebook 2), la posibilidad de usar un framework si lo necesitas (notebook 3), RAG si tu reto requiere consultar documentos propios (notebook 4), y MCP si quieres exponer tus herramientas de forma estandarizada (notebook 5).

El proceso de diseño de un agente sigue siempre el mismo orden:
1. **Definir el caso de uso**: problema real, usuario, criterio de éxito.
2. **Diseñar las herramientas**: qué acciones o consultas necesita el agente, con una `description` clara para cada una.
3. **Implementar el loop**: razonar → actuar → observar, hasta una respuesta final.
4. **Probar**: casos de prueba concretos, no solo "que funcione una vez".
5. **Reflexionar**: qué riesgos tiene (loops infinitos, alucinación de herramientas, costos) y qué nivel de autonomía es el apropiado para este caso.

Este notebook usa deliberadamente el patrón más simple (el loop manual, sin framework) para que ningún equipo se bloquee por un problema de instalación durante el taller — agreguen complejidad (RAG, MCP, LangChain) solo si su reto realmente lo necesita.


## 0. Instalación (base mínima)

In [ ]:
!pip install -q google-genai

### Configurar API key de Gemini

**Cómo obtenerla:** [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (gratis, dos clics).

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `GEMINI_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.

⚠️ **Aviso conocido (2026):** Google está migrando las API keys al nuevo formato con prefijo `AQ.` (antes `AIza...`). Hay reportes activos y aún no resueltos en el foro oficial de Google de que las keys `AQ.` devuelven `401 ACCESS_TOKEN_TYPE_UNSUPPORTED` en algunas cuentas/proyectos, incluso bien configuradas. La celda de abajo te dice qué tipo de key tienes para descartar esto como causa del error.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass("Pega tu GEMINI_API_KEY: ")

_key = os.environ.get("GEMINI_API_KEY", "")
print("API key configurada:", "OK" if _key else "FALTA")

if _key.startswith("AQ."):
    print("ADVERTENCIA: tu key tiene el nuevo formato 'AQ.' -- si mas adelante ves un error 401")
    print("ACCESS_TOKEN_TYPE_UNSUPPORTED, es un problema conocido y actualmente activo del lado de")
    print("Google con este formato de key, no de este notebook. Revisa:")
    print("https://discuss.ai.google.dev/c/gemini-api/4  (buscar 'AQ. 401 ACCESS_TOKEN_TYPE_UNSUPPORTED')")
elif _key.startswith("AIza"):
    print("Formato de key clasico (AIza...) -- no deberia verse afectado por el problema de las keys 'AQ.'.")


## 1. El loop del agente (reutilizado del notebook 2)

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])  # explícito: evita depender de la autodetección de entorno
MODEL = "gemini-3.5-flash"

def run_agent(user_message: str, tools, implementations, max_steps: int = 5, verbose: bool = True):
    """Implementación manual del ciclo ReAct sobre la Interactions API de Gemini."""
    interaction = client.interactions.create(model=MODEL, input=user_message, tools=tools)

    for step_num in range(1, max_steps + 1):
        function_calls = [s for s in interaction.steps if s.type == "function_call"]

        if not function_calls:
            if verbose:
                print(f"[Paso {step_num}] Respuesta final del agente.")
            return interaction.output_text

        function_results = []
        for fc in function_calls:
            if verbose:
                print(f"[Paso {step_num}] Actuando: {fc.name}({fc.arguments})")
            fn = implementations.get(fc.name)
            result = fn(**fc.arguments) if fn else f"Herramienta desconocida: {fc.name}"
            if verbose:
                print(f"[Paso {step_num}] Observando resultado: {result}")
            function_results.append({
                "type": "function_result",
                "name": fc.name,
                "call_id": fc.id,
                "result": [{"type": "text", "text": str(result)}],
            })

        interaction = client.interactions.create(
            model=MODEL, input=function_results, tools=tools,
            previous_interaction_id=interaction.id,
        )

    return "Se alcanzó el límite de pasos (max_steps) sin una respuesta final."


## 2. Paso 1 — Define tu caso de uso

Completa esto con tu equipo antes de programar:


In [ ]:
caso_de_uso = {
    "problema": "TODO: ¿qué problema real de la universidad resuelve tu agente?",
    "usuario": "TODO: ¿quién lo usaría? (estudiante, profesor, administrativo)",
    "exito": "TODO: ¿cómo sabes que el agente respondió bien?",
}
caso_de_uso


## 3. Paso 2 — Define las herramientas que tu agente necesitará

Sigue el mismo patrón: cada herramienta necesita `type: "function"`, un `name`, una `description` clara (de esto depende que el modelo la use bien) y una implementación real.


In [ ]:
# TODO: reemplaza estas herramientas de ejemplo por las de tu reto

def herramienta_1(parametro: str) -> str:
    """TODO: implementa la lógica real (consulta a una API, a una base de datos, etc.)"""
    return f"Resultado simulado para: {parametro}"

herramientas_taller = [
    {
        "type": "function",
        "name": "herramienta_1",
        "description": "TODO: describe cuándo debe usarse esta herramienta",
        "parameters": {
            "type": "object",
            "properties": {"parametro": {"type": "string"}},
            "required": ["parametro"]
        }
    },
]

implementaciones_taller = {
    "herramienta_1": herramienta_1,
}


## 4. Paso 3 — Ejecuta el agente con tus herramientas

In [ ]:
respuesta = run_agent(
    "TODO: escribe aquí una pregunta real de prueba para tu agente",
    herramientas_taller,
    implementaciones_taller,
)
print(respuesta)


## 5. Paso 4 — Casos de prueba (mínimo 3, según el checklist de entrega)

In [ ]:
casos_de_prueba = [
    {"pregunta": "TODO caso 1", "resultado_esperado": "TODO", "resultado_obtenido": None},
    {"pregunta": "TODO caso 2", "resultado_esperado": "TODO", "resultado_obtenido": None},
    {"pregunta": "TODO caso 3", "resultado_esperado": "TODO", "resultado_obtenido": None},
]

for caso in casos_de_prueba:
    caso["resultado_obtenido"] = run_agent(caso["pregunta"], herramientas_taller, implementaciones_taller, verbose=False)

for caso in casos_de_prueba:
    print("Pregunta:", caso["pregunta"])
    print("Resultado:", caso["resultado_obtenido"])
    print("---")


## 6. Paso 5 — Reflexión final (entregable)

Responde brevemente, en base a lo visto en la charla:

1. **¿Qué límites o riesgos identificaron?** (loops infinitos, alucinación de herramientas, costos, datos sensibles...)
2. **¿Qué nivel de autonomía eligieron para el agente?** (asistido, human-in-the-loop, supervisado, autónomo) ¿Por qué?
3. **¿Qué agregarían si tuvieran una sesión más?** (ej: evaluación automática, observabilidad, exponerlo vía MCP)

*Estos tres puntos se profundizan en la Sesión 2 (Evaluación, Observabilidad) y en el Taller 6 (Seguridad y costos).*
